## 강화학습 3장. Dynamic Programming

* 출처 1. https://with-rl.tistory.com/entry/%EB%B0%94%EB%8B%A5%EB%B6%80%ED%84%B0-%EB%B0%B0%EC%9A%B0%EB%8A%94-%EA%B0%95%ED%99%94-%ED%95%99%EC%8A%B5-04-MDP%EB%A5%BC-%EC%95%8C-%EB%95%8C%EC%9D%98-%ED%94%8C%EB%9E%98%EB%8B%9D
* 출처 2. https://github.com/seungeunrho/RLfrombasics/tree/master

In [5]:
import gymnasium as gym
import numpy as np

---

### 3.1. 본 수업 : OpenAI Gym의 FrozenLake-v1 환경에서 정책 평가와 정책 반복 알고리즘을 구현

In [6]:
# Temporal Difference Learning [TD]

"""
구현을 위해 필요한 요소들
1. Grid World 환경 직접 구현 → gymnasium FrozenLake-v1 환경 활용(바닥이 미끄러운 얼음 호수 건너기)
2. TD / MC 밸류 테이블 채우기 → 4 by 4 격자
3. 랜덤 정책(밸류에서 정책 뽑기, 어느 방향이 제일 높은지 argmax) → 정책 평가 
4. 정책 반복(평가, 개선, 평가, 개선, ...)
5. TD / MC  → MDP 모델이 없는 경우(모르는 경우) 학습
"""

# Monte Carlo [MC]

'\n구현을 위해 필요한 요소들\n1. Grid World 환경 직접 구현 → gymnasium FrozenLake-v1 환경 활용(바닥이 미끄러운 얼음 호수 건너기)\n2. TD / MC 밸류 테이블 채우기 → 4 by 4 격자\n3. 랜덤 정책(밸류에서 정책 뽑기, 어느 방향이 제일 높은지 argmax) → 정책 평가 \n4. 정책 반복(평가, 개선, 평가, 개선, ...)\n5. TD / MC  → MDP 모델이 없는 경우(모르는 경우) 학습\n'

---

### 3.2. 벨만 방정식과 정책 평가, 정책 반복(개선) 알고리즘 [결정론적]
* 반복적 정책 평가(iterative policy evaluation) 방법을 통해 각 상태 $s$에 대한 가치 함수 $v(s)$ 계산 가능

In [11]:
gamma: float = 0.9
# 할인율

def policyEvaluation(env, policy):
    """랜덤 정책을 벨만 방정식을 이용하여 평가하는 메서드
    args :
    policy 정책 pi

    environment : 
    Frozen(F) 호수 위를 걸어가면서 Holes(H)에 빠지지 않고 
    Start(S)에서 골(G)까지 얼어붙은 호수를 건너는 것을 목표

    logics:
    정책 반복 알고리즘

    returns :
    V 가치함수 v_n array
    """
    V = np.zeros( env.observation_space.n ) 
    # 가치함수를 저장할 표 생성

    while True:
        oldV = V.copy()
        # 현재 가치함수를 oldV에 복사하여 저장(이전 가치함수)

        for state in range( env.observation_space.n ):
            # 각 상태에 대해
            v = 0
            # 현재 상태에서의 가치 초기화

            for action, action_prob in enumerate(policy[state]):
                # 벨만 방정식(순환식)
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률
                v = v  + action_prob(reward + gamma * V[next_state])
                # [핵심] 현재 상태의 가치 = 현재 상태에서 특정 행동을 할 확률(왼오위아래) x (보상 + 감쇠인자 * 다음 상태의 가치)
            
            V[state] = v
            # 현재 상태의 가치(값) 표에 저장

        if max( np.abs(V - oldV) ) < 1e-8:
            # [핵심] 변화가 충분히 작으면 수렴으로 간주(self-consistency, bootstrapping)
            break

### 3.3.1. 벨만 최적 방정식과 정책 반복 알고리즘 [결정론적]

In [12]:
gamma: float = 0.9
# 할인율

def policyIteration(env):
    """랜덤 정책을 벨만 방정식을 이용하여 평가하고 개선 단계를 반복하여 최적 정책으로 접근하는 메서드
    args:
    env 과업

    environment : 
    Frozen(F) 호수 위를 걸어가면서 Holes(H)에 빠지지 않고 
    Start(S)에서 골(G)까지 얼어붙은 호수를 건너는 것을 목표

    logics:
    정책 반복 알고리즘

    returns :
    - V 최적가치함수 v_n
    - pi 최적정책 pi*
    """
    V = np.zeros( env.observation_space.n )
    # 가치함수를 저장할 표 생성
    pi = [0 for i in range( env.observation_space.n )]
    # 정책을 행동 0으로 초기화

    while True:
        # 1+2. 정책 반복(Generalized Policy Iteration strategy)
        while True:
            # 1. 정책 평가 단계(Evaluation)
            oldV = V.copy()
            # 현재 가치함수를 oldV에 복사하여 저장(이전 가치함수)

            for state in range( env.observation_space.n ):
                # 각 상태에 대해
                action = pi[state]
                # [핵심] 현재 상태에서의 최적의 행동을 할 확률(최적 정책)
                # [as-is] for action, action_prob in enumerate(policy[state]): # 벨만 방정식(순환식)
                
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률
                v = reward + gamma * V[next_state]
                # [핵심] 현재 상태의 가치 = (보상 + 감쇠인자 * 다음 상태의 가치)
                
                V[state] = v
                # 현재 상태의 가치(값) 표에 저장

            if max( np.abs(V - oldV) ) < 1e-8:
                # [핵심] 변화가 충분히 작으면 수렴으로 간주(self-consistency, bootstrapping)
                break
        
        converged = True
        # 2. 정책 개선 단계(Iteration)
        for state in range( env.observation_space.n ):
            # 각 상태에 대해
            old_action = pi[state]
            # 
            q = np.zeros( env.action_space.n )
            # 탐욕적인 선택(greedy), 정책을 2차원이 아닌 1차원 표에 저장

            for action in range( env.action_space.n ):
            # 각 행동 공간에서의 행동에 대해
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률
                q[action] = reward + gamma * V[next_state]
                # [핵심] 기댓값 연산 = (보상 + 감쇠인자 * 다음 상태의 가치)
                if pi[state] != old_action:
                    converged = False
                    # 변화가 없으면 수렴으로 간주하고 루프 중단

                if converged :
                    break
            return pi, V
            # 최적 정책과 최적 가치 함수 반환

### 3.3.2. 벨만 최적 방정식과 가치 반복 알고리즘 [결정론적]

In [13]:
gamma: float = 0.9
# 할인율

def valueIteration(env):
    """정책 없이 벨만 방정식을 이용하여 가치 개선 단계를 반복하여 최적 정책으로 접근하는 메서드
    args:
    env 과업

    environment : 
    Frozen(F) 호수 위를 걸어가면서 Holes(H)에 빠지지 않고 
    Start(S)에서 골(G)까지 얼어붙은 호수를 건너는 것을 목표

    logics:
    가치 반복 알고리즘

    returns :
    - V 최적가치함수 v_n
    - pi 최적정책 pi*
    """
    V = np.zeros(env.observation_space.n) 
    # 가치함수를 저장할 표 생성
    
    while True:
    # 1. 가치 개선 단계
        oldV = V.copy()
        # 현재 가치함수를 oldV에 복사하여 저장(이전 가치함수)
        
        for state in range( env.observation_space.n ):
        # 각 상태에 대해
            q = np.zeros( env.action_space.n )
            # 탐욕적인 선택(greedy), 정책을 2차원이 아닌 1차원 표에 저장

            for action in range( env.action_space.n ):
            # 각 행동 공간에서의 행동에 대해
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률
                q[action] = reward + gamma * V[next_state]
                # [핵심] 기댓값 연산 = (보상 + 감쇠인자 * 다음 상태의 가치)
                V[state] = np.max(q)
                # [핵심] q의 최댓값을 표에 저장

            if max( np.abs(V - oldV) ) < 1e-8:
                # [핵심] 변화가 충분히 작으면 수렴으로 간주(self-consistency, bootstrapping)
                break
        
        pi = env.observation_space.n * [None]
        # 2. 최적 가치 함수로부터 최적 정책 구하는 단계
        for state in range( env.observation_space.n ):
            # 각 상태에 대해
            q = np.zeros( env.action_space.n )
            # 정책 저장할 표 생성

            for action in range( env.action_space.n ):
            # 각 행동 공간에서의 행동에 대해
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률
                q[action] = reward + gamma * V[next_state]
                # [핵심] 기댓값 연산 = (보상 + 감쇠인자 * 다음 상태의 가치)

            pi[state] = np.argmax(q)
            # [핵심] 최적 정책
        return pi, V
        # 최적 정책과 최적 가치 함수 반환

### 3.4. 스토캐스틱 MDP를 위한 가치 반복 알고리즘

In [14]:
gamma: float = 0.9
# 할인율

def stocasticValueIteration(env):
    """스토캐스틱 환경에서 정책 없이 벨만 방정식을 이용하여 가치 개선 단계를 반복하여 최적 정책으로 접근하는 메서드
    args:
    env 과업

    environment : 
    Frozen(F) 호수 위를 걸어가면서 Holes(H)에 빠지지 않고 
    Start(S)에서 골(G)까지 얼어붙은 호수를 건너는 것을 목표

    logics:
    MDP를 모르는 환경에서 가치 반복 알고리즘

    returns :
    - V 최적가치함수 v_n
    - pi 최적정책 pi*
    """
    V = np.zeros(env.observation_space.n) 
    # 가치함수를 저장할 표 생성
    
    while True:
    # 1. 가치 개선 단계
        oldV = V.copy()
        # 현재 가치함수를 oldV에 복사하여 저장(이전 가치함수)
        
        for state in range( env.observation_space.n ):
        # 각 상태에 대해
            q = np.zeros( env.action_space.n )
            # 탐욕적인 선택(greedy), 정책을 2차원이 아닌 1차원 표에 저장

            for action in range( env.action_space.n ):
            # 각 행동 공간에서의 행동에 대해
                for prob, next_state, reward, terminated in env.unwrapped.P[state][action]:
                # [핵심] 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                    q[action] = q[action] + prob(reward + gamma * V[next_state])
                    # [핵심] 기댓값 연산 = 확률 * (보상 + 감쇠인자 * 다음 상태의 가치)
                V[state] = np.max(q)
                # q의 최댓값을 표에 저장

            if max( np.abs(V - oldV) ) < 1e-8:
                # 변화가 충분히 작으면 수렴으로 간주(self-consistency, bootstrapping)
                break
        
        pi = env.observation_space.n * [None]
        # 2. 최적 가치 함수로부터 최적 정책 구하는 단계
        for state in range( env.observation_space.n ):
            # 각 상태에 대해
            q = np.zeros( env.action_space.n )
            # 정책 저장할 표 생성

            for action in range( env.action_space.n ):
            # 각 행동 공간에서의 행동에 대해
                for prob, next_state, reward, terminated in env.unwrapped.P[state][action]:
                # [핵심]
                    q[action] = q[action] + prob(reward + gamma * V[next_state])
                    # [핵심] 기댓값 연산 = 확률 * (보상 + 감쇠인자 * 다음 상태의 가치

            pi[state] = np.argmax(q)
            # [핵심] 최적 정책
        return pi, V
        # 최적 정책과 최적 가치 함수 반환

### 3.2~3.4. 결과 비교 [as-is]

In [17]:
env = gym.make(
    "FrozenLake-v1",
    is_slippery = False,
    render_mode = "ansi"
) # determine

In [ ]:
pi_01 = np.ones((env.observation_space, env.action_space.n)) / env.action_space.n
# 랜덤 정책
V_01 = policyEvaluation(env, pi_01)

In [ ]:
pi_02, V_02= policyIteration(env)

In [ ]:
pi_03, V_03= valueIteration(env)

In [ ]:
env = gym.make(
    "FrozenLake-v1",
    is_spippery = True,
    render_mode = "ansi"
) # stocastic

In [ ]:
pi_04, V_04 = stocasticValueIteration(env)